# Transient Natural-Circulation Loop
### Coupled momentum and thermal storage in passive cooling

**Learning goal:** Build a reduced-order natural-circulation model from control-volume balances, simulate startup after a heat-input step, verify global energy conservation and the steady state, and explain the nonlinear scaling between heat load, temperature rise, and circulation rate.

**Time:** 60-90 minutes. Python, NumPy, SciPy, Matplotlib. No spatial mesh or grid-refinement study is used.

**Physical intuition:** Heating makes one vertical leg lighter than the cooled leg. The resulting buoyancy head accelerates the loop, while form/friction losses grow approximately as velocity squared. Flow then transports heat from the hot control volume to the cold one, reducing the temperature difference that created the flow. This feedback can overshoot or oscillate before settling.

**Success targets:** Reproduce the analytical steady state to 1%; close the integrated external energy balance to 0.1%; recover the predicted mass-flow power-law exponent within 0.03; and distinguish the full inertial model from a quasi-steady momentum closure.

The notebook is an assignment. Implementation cells contain TODOs and no completed model.

## 1. Model, assumptions, and sign convention

Represent a single-phase rectangular loop using three states: signed mean loop velocity $v$, hot-leg mean temperature $T_h$, and cold-leg mean temperature $T_c$. Assume constant properties, incompressible Boussinesq buoyancy, two well-mixed thermal control volumes, a single quadratic loss coefficient, and one heat sink. Positive $v$ is the buoyancy-preferred direction.

$$\frac{dv}{dt}=\frac{g\beta H}{L}(T_h-T_c)-\frac{K}{2L}v|v|,$$
$$C_h\frac{dT_h}{dt}=Q(t)-|\dot m|c_p(T_h-T_c),$$
$$C_c\frac{dT_c}{dt}=|\dot m|c_p(T_h-T_c)-UA(T_c-T_s),$$
with $\dot m=\rho A v$, $C_h=M_hc_p$, and $C_c=M_cc_p$.

The absolute value in the advective heat-transfer magnitude is a deliberate two-node closure: either circulation direction transports energy between the heated and cooled regions. It is not a general substitute for signed advection in a spatial system code. Use the signed $v$ in the momentum equation and report any reversal.

Base case: water-like constant properties, $\rho=997$ kg/m3, $c_p=4180$ J/(kg K), $\beta=2.1\times10^{-4}$ 1/K, $A=2.0\times10^{-3}$ m2, $L=8$ m, $H=2.5$ m, $K=20$, $M_h=M_c=10$ kg, $UA=120$ W/K, $T_s=300$ K, and a 5 kW heat step at $t=0$. Integrate to 3600 s from $(v,T_h,T_c)=(0,300,300)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar

PARAMS = {
    "rho": 997.0, "cp": 4180.0, "beta": 2.1e-4,
    "g": 9.81, "A": 2.0e-3, "L": 8.0, "H": 2.5,
    "K": 20.0, "M_h": 10.0, "M_c": 10.0,
    "UA": 120.0, "T_sink": 300.0,
}
Q_BASE = 5_000.0
T_SPAN = (0.0, 3600.0)
T_EVAL = np.linspace(*T_SPAN, 1801)
Y0 = np.array([0.0, 300.0, 300.0])

# TODO: Check units, record assumptions, and configure plotting defaults.


## 2. Implement the dynamic and quasi-steady models

Write `rhs_full(t, y, p, qfun, uafun)` for the three-state model. Compute every heat rate explicitly and return derivatives in SI units. Keep $v|v|$ rather than $v^2$ so the drag always opposes motion.

For comparison, eliminate momentum inertia using
$$v_{qs}=\operatorname{sign}(\Delta T)\sqrt{\frac{2g\beta H|\Delta T|}{K}},\qquad \Delta T=T_h-T_c,$$
with $v_{qs}=0$ at $\Delta T=0$. Integrate only the two energy equations with this algebraic velocity. Use `solve_ivp` with `method="Radau"`, `rtol=1e-8`, and `atol=[1e-10,1e-8,1e-8]` for the full model; use compatible temperature tolerances for the quasi-steady model.

Do not clip temperatures, velocity, or derivatives. A nonphysical result should fail a diagnostic rather than be hidden.

In [ ]:
def heat_input(t, q=Q_BASE):
    # TODO: Return the prescribed step heat input.
    raise NotImplementedError

def ua_base(t, ua=PARAMS["UA"]):
    # TODO: Return the base heat-sink conductance.
    raise NotImplementedError

def rhs_full(t, y, p, qfun=heat_input, uafun=ua_base):
    """Return [dv/dt, dTh/dt, dTc/dt]."""
    # TODO: Implement momentum and two energy balances.
    raise NotImplementedError

def quasi_steady_velocity(delta_t, p):
    # TODO: Implement the signed algebraic momentum closure.
    raise NotImplementedError

def rhs_quasi(t, y, p, qfun=heat_input, uafun=ua_base):
    """Return [dTh/dt, dTc/dt] using the algebraic velocity."""
    # TODO: Implement using the same heat-rate definitions.
    raise NotImplementedError


## 3. Analytical steady state

For constant positive $Q$ and $T_h>T_c$, set all derivatives to zero. Show that
$$T_c^*=T_s+\frac{Q}{UA},$$
$$\Delta T^*=\left[\frac{Q}{\rho A c_p\sqrt{2g\beta H/K}}\right]^{2/3},$$
$$v^*=\sqrt{\frac{2g\beta H\Delta T^*}{K}},\qquad \dot m^*=\rho A v^*,$$
and $T_h^*=T_c^*+\Delta T^*$.

Derive the relations before coding them. Evaluate the residual of all three steady equations at the analytical state. The result should be roundoff-small relative to the characteristic acceleration and heat rates.

Predict the power laws $\Delta T^*\propto Q^{2/3}$ and $\dot m^*\propto Q^{1/3}$. These are model scalings, not numerical convergence orders.

In [ ]:
def analytical_steady(q, p):
    # TODO: Return v_star, Th_star, Tc_star, mdot_star.
    raise NotImplementedError

# TODO: Compute the base analytical state and evaluate rhs_full there.


## 4. Startup transient and diagnostics

Integrate the full and quasi-steady models on the same output times. Plot:

1. $T_h$, $T_c$, and the analytical steady temperatures.
2. Signed mass flow $\dot m(t)$ with its analytical steady value.
3. Buoyancy acceleration $a_b=g\beta H(T_h-T_c)/L$ and drag acceleration $a_d=Kv|v|/(2L)$.
4. Full-model minus quasi-steady $T_h$, $T_c$, and mass flow.

Report 10-90% rise time, maximum hot-leg overshoot relative to $T_h^*$, time to enter and remain within 1% of each steady state, and whether $v$ reverses. Define each metric carefully. Do not claim that a two-node model predicts real plant stability.

The model-wide energy identity is
$$\frac{d}{dt}[C_h(T_h-T_s)+C_c(T_c-T_s)]=Q-UA(T_c-T_s).$$
Numerically integrate the right side with the trapezoidal rule and compare it with stored-energy change. Normalize by the larger of total input energy and 1 J.

In [ ]:
# TODO: Solve both models, compute time histories and transient metrics,
# and evaluate the integrated external energy-balance closure.


## 5. Heat-load sweep and nonlinear scaling

Repeat the full transient for $Q\in\{1,2.5,5,10\}$ kW with all physical parameters and final time unchanged. For each case, tabulate final and analytical values of $\dot m$, $T_h$, $T_c$, plus relative steady-state errors.

Fit straight lines to $(\log Q,\log \dot m^*)$ and $(\log Q,\log \Delta T^*)$. Compare fitted slopes with $1/3$ and $2/3$. Use analytical steady values for the primary scaling fit and the final transient values only as an independent check that 3600 s is sufficient.

Plot mass flow and hot-minus-cold temperature versus heat load on log-log axes with the predicted slopes. Explain why $T_c-T_s=Q/UA$ scales linearly even though $T_h-T_c$ does not.

In [ ]:
# TODO: Run the four-load sweep, create the table, fit the two exponents,
# and produce the required log-log plot with labeled reference slopes.


## 6. Automatic verification and validity checks

Assert:

- Every solve succeeds and returns finite states on the complete requested interval.
- Density, heat capacity, geometry, loss coefficient, conductance, and thermal masses are positive.
- Base-case final $v$, $T_h$, and $T_c$ are each within 1% of the analytical steady state.
- The maximum normalized integrated external energy imbalance is below $10^{-3}$.
- The analytical steady state makes each governing-equation residual negligible: heat-rate residuals below $10^{-9}\max(Q,1)$ and acceleration residual below $10^{-12}$ m/s2.
- The fitted analytical exponents differ from $1/3$ and $2/3$ by less than 0.03.
- The Boussinesq indicator $\beta\max|T_h-T_c|$ is reported and below 0.05 for every base/sweep case.
- No temperature falls below 250 K or exceeds 450 K in this synthetic study.

Do not assert that the full model must overshoot, oscillate, or reverse; those are outcomes. Do not loosen tolerances to conceal a bookkeeping or sign error.

In [ ]:
# TODO: Implement all assertions and print a compact pass/fail summary.


## 7. Interpretation, stretch goal, and references

Answer briefly:

1. Which feedback closes the loop between buoyancy and heat transport?
2. When is the quasi-steady momentum closure adequate, and what transient information does it remove?
3. Why does quadratic drag produce the $Q^{1/3}$ flow scaling in this closure?
4. Which assumptions would break first in a real decay-heat removal loop: constant properties, two thermal nodes, fixed $K$, single-phase flow, or instantaneous mixing?
5. What additional states and closure laws would a GOTHIC/TRACE-style system model need?

**Stretch goal (15-25 minutes):** At $t=1800$ s, reduce $UA$ from 120 to 40 W/K while holding $Q=5$ kW. Simulate the heat-sink-degradation transient, plot the new equilibrium approach, and verify the same global energy balance with time-dependent $UA(t)$. Predict which steady quantities change before running.

### References

- **Dedicated project reference:** International Atomic Energy Agency, *Natural Circulation in Water Cooled Nuclear Power Plants*, IAEA-TECDOC-1474, Vienna, 2005. [Open PDF](https://www-pub.iaea.org/MTCD/Publications/PDF/TE_1474_web.pdf). Focus on Section 4.2 and the single-phase hot-/cold-region balance model; compare its assumptions with today's three-state closure.
- **Solver reference:** [SciPy `solve_ivp` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html). Record SciPy version, method, tolerances, and solver status.
- **Paper to read:** Dean Wilson, Hector Iacovides, and Timothy J. Craft, "Numerical insights into the transient behaviour of single-phase Natural Circulation Loops for nuclear passive cooling applications," *Annals of Nuclear Energy*, 208, 110730, 2024. DOI: 10.1016/j.anucene.2024.110730. [Open-access article](https://doi.org/10.1016/j.anucene.2024.110730); [author manuscript](https://papers.ssrn.com/sol3/Delivery.cfm/49d50cad-eaf4-4b2a-8ef7-e4bee84683d5-MECA.pdf?abstractid=4646390&mirid=1). Read for how Rayleigh number, Prandtl number, aspect ratio, oscillation, and reversal emerge in 2D URANS beyond a lumped model.

### Topic ledger

| Date / record | Primary objective | Status |
|---|---|---|
| Older assignments, dates unavailable | Grid convergence and errors | User reported five repetitive assignments; individual records unavailable |
| 2026-10-05 | Grid convergence and discretization uncertainty | Inspected |
| 2026-10-06 corrected | Fixed-mesh advection, CFL, conservation, positivity, TV | Inspected |
| 2026-10-07 | Krylov methods and line-block preconditioning | Inspected |
| 2026-10-08 | Data layout, cache behavior, and kernel throughput | Inspected |
| 2026-10-09 | Nonlinear natural-circulation thermal-fluid modeling | This assignment |

Only four preceding distinct projects are recoverable, so a complete seven-project audit remains unavailable. The reported older repetitions remain excluded. This is assignment 3 of the ten-assignment exclusion established October 6; seven remain after today. Planned next primary objectives: optimization, multiphysics coupling, high-order reconstruction, then parallel algorithms. No grid-refinement, observed-order, Richardson, or GCI task appears here.